# 02 — Build and Inspect the Job Vector Index

This notebook demonstrates the index-building workflow used by SmartHire. The production embedding provider is selected through configuration, so the current default is Gemini API embeddings. On Windows, the resulting vectors are stored and searched with the NumPy fallback when native FAISS is unavailable.

In [1]:
from pathlib import Path
import sys

# Find the SmartHire project root so these notebooks work whether Jupyter
# is launched from the project root or from the notebooks/ directory.
HERE = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "src" / "config.py").exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find the SmartHire project root. "
        "Open this notebook from inside the project repository."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: C:\Users\harsh\Downloads\Final Gen ai project output\smarthire-genai-final


## Step 1 — Inspect the source dataset and configuration

The job CSV is the source dataset. The vectorstore is a generated artifact that can be rebuilt from the dataset.

In [2]:
import json
from src import config
from src.core.paths import JOBS_CSV, EMBEDDING_META, FAISS_INDEX
from src.data.loader import load_jobs

jobs = load_jobs(JOBS_CSV)

print("Job dataset:", JOBS_CSV)
print("Jobs loaded:", len(jobs))
print("Embedding provider:", config.EMBEDDING_PROVIDER)
print("Gemini embedding model:", config.GEMINI_EMBEDDING_MODEL)
print("Embedding batch size:", config.EMBEDDING_BATCH_SIZE)

Job dataset: C:\Users\harsh\Downloads\Final Gen ai project output\smarthire-genai-final\data\jobs\smarthire_jobs_80.csv
Jobs loaded: 80
Embedding provider: gemini-api
Gemini embedding model: gemini-embedding-2
Embedding batch size: 64


## Step 2 — Rebuild the index

The call below is the same rebuild function used by `scripts/build_index.py`. Timing the call makes the notebook useful for comparing index-build cost after changes to the dataset or embedding provider.

In [3]:
import time
from src.search.job_search import rebuild_index

start = time.perf_counter()
result = rebuild_index()
elapsed = time.perf_counter() - start

print(result["message"])
print(f"Build time: {elapsed:.2f} seconds")

Index rebuilt for 80 jobs using NumPy exact cosine.
Build time: 6.41 seconds


## Step 3 — Inspect the generated metadata

The metadata records which embedding provider actually produced the vectors. This is important because the query embeddings must live in the same vector space as the stored job embeddings.

In [4]:
if EMBEDDING_META.exists():
    metadata = json.loads(
        EMBEDDING_META.read_text(encoding="utf-8")
    )
    print(json.dumps(metadata, indent=2))
else:
    print("Embedding metadata file was not created.")

{
  "provider": "gemini-api",
  "model": "gemini-embedding-2",
  "dimension": 768,
  "batch_size": 64
}


## Step 4 — Verify the final index artifact

On a Windows development machine, the application may use an `.npy` vector matrix and exact NumPy cosine search. In a Linux deployment with `faiss-cpu` installed, the same vectors can be placed in a FAISS index.

In [5]:
npy_path = FAISS_INDEX.with_suffix(".npy")

print("FAISS index exists:", FAISS_INDEX.exists())
print("NumPy vector artifact exists:", npy_path.exists())

if npy_path.exists():
    import numpy as np
    vectors = np.load(npy_path)
    print("Stored vector matrix shape:", vectors.shape)

print()
print("Index build completed successfully.")

FAISS index exists: False
NumPy vector artifact exists: True
Stored vector matrix shape: (80, 768)

Index build completed successfully.


## After the build

Record the reported number of jobs and build time in your project notes if you are comparing experiments.

The important property for the deployed application is reproducibility: the repository's job CSV can be used to regenerate the vector index inside the runtime when a generated artifact is absent.